# PINN 2 — 1D viscous Burgers' equation

$$u_t + u\,u_x = \nu\,u_{xx}, \qquad x\in[-1,1],\; t\in[0,1],\; \nu = 0.01/\pi$$

$$u(x,0) = -\sin(\pi x), \qquad u(-1,t) = u(1,t) = 0.$$

New compared with PINN 1: two inputs $(x,t)$, an **initial condition**, and a
**nonlinear** convection term that steepens the solution into a near-shock at $x=0$.

The reference solution is computed with the **Cole–Hopf transformation**
(`neural_pde_solvers.pinn.burgers_reference`), so we can report a quantitative error.

| | |
|---|---|
| Problem class | `neural_pde_solvers.pinn.Burgers1D` |
| Config | `configs/pinn/burgers1d.yaml` |
| CLI | `pinn-run configs/pinn/burgers1d.yaml` |

In [ ]:
# --- Setup -------------------------------------------------------------------
# Locally: `pip install -e ".[pinn]"` from the repository root.
# On Google Colab the package is installed from GitHub automatically.
import importlib.util
import os
import subprocess
import sys

if importlib.util.find_spec("neural_pde_solvers") is None:
    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "neural-pde-solvers[pinn] @ git+https://github.com/tvh0ng/neural-pde-solvers.git",
        ]
    )

import matplotlib.pyplot as plt
import numpy as np
import torch

from neural_pde_solvers import set_seed, setup_logging
from neural_pde_solvers.pinn import (
    MLP,
    PINNTrainer,
    TrainerConfig,
    count_parameters,
    evaluate,
    grid,
    predict,
    resolve_device,
)
from neural_pde_solvers.pinn.plotting import plot_loss_history

setup_logging()
set_seed(42)
device = resolve_device("auto")
QUICK = os.environ.get("PINN_QUICK") == "1"  # shortened run used by CI
print(f"PyTorch {torch.__version__} | device: {device}")

## 1. Problem

Residual (`Burgers1D.residual`):

```python
u   = model(X)                     # X = [x, t]
du  = gradient(u, X)               # [u_x, u_t]
u_xx = gradient(du[:, :1], X)[:, :1]
return du[:, 1:] + u * du[:, :1] - nu * u_xx
```

Loss $= \text{pde} + 20\,\text{ic} + 10\,\text{bc}$ — the initial condition carries the
whole time evolution, so it gets the largest weight.

In [ ]:
from neural_pde_solvers.pinn import Burgers1D

problem = Burgers1D(nu=0.01 / np.pi, n_ic=200, n_bc=200).to(device)
print("domain:", problem.domain(), "| weights:", problem.default_weights)

## 2. Network

`normalize_inputs` maps $(x,t)$ affinely to $[-1,1]^2$ inside the network
(`MLP(lb=..., ub=...)`), which keeps all inputs on the same scale.

In [ ]:
lb, ub = problem.domain()
model = MLP(in_dim=2, width=64, depth=4, activation="tanh", lb=lb, ub=ub).to(device)
print(f"trainable parameters: {count_parameters(model):,}")

## 3. Training — Adam, then L-BFGS

Adam (`StepLR`, halved every 2000 epochs) finds the right basin; a short **L-BFGS**
stage with strong-Wolfe line search then sharpens the shock front considerably.

In [ ]:
config = TrainerConfig(
    epochs=500 if QUICK else 10000,
    lr=1e-3,
    n_collocation=2000 if QUICK else 10000,
    scheduler="step",
    scheduler_kwargs=dict(step_size=2000, gamma=0.5),
    lbfgs_iters=50 if QUICK else 2000,
    log_every=1000,
)
history = PINNTrainer(model, problem, config).fit()

## 4. Results

In [ ]:
from neural_pde_solvers.pinn.plotting import plot_prediction_vs_exact_2d

mesh, X = grid(problem, (256, 101))
u_pred, u_ref = predict(model, X), problem.exact(X)
metrics = evaluate(model, problem, (256, 101))
print(metrics)

fig, _ = plot_prediction_vs_exact_2d(mesh, u_pred, u_ref, labels=("x", "t"), cmap="RdBu_r")
plt.show()

In [ ]:
Xm, Tm = mesh
U, R = u_pred.reshape(Xm.shape), u_ref.reshape(Xm.shape)
fig, axes = plt.subplots(1, 4, figsize=(19, 4), sharey=True)
for ax, t in zip(axes[:3], (0.25, 0.5, 0.75), strict=True):
    k = int(round(t * (Xm.shape[0] - 1)))
    ax.plot(Xm[k], R[k], lw=2, label="Cole–Hopf")
    ax.plot(Xm[k], U[k], "--", lw=2, label="PINN")
    ax.set(title=f"t = {t}", xlabel="x")
    ax.grid(True, alpha=0.3)
axes[0].set_ylabel("u(x, t)")
axes[0].legend()
plot_loss_history(history, ax=axes[3])
fig.tight_layout()
plt.show()

## Key takeaways

1. A time-dependent PDE is just another input dimension for the network.
2. Three loss terms (PDE, IC, BC); loss weighting matters, the IC most of all.
3. The steep front at $x=0$ is where the error concentrates — L-BFGS refinement,
   more collocation points near the front, or adaptive weighting all help.
4. Always compare with a trustworthy reference (here, Cole–Hopf) rather than eyeballing.